In [1]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

Data Ingestion

In [3]:
from langchain.document_loaders import TextLoader

In [4]:
loader = TextLoader("../data/agenticAI.txt", encoding="utf8")
documents = loader.load()

In [5]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'**Understanding Agentic AI**\n\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\n\n**Key Characteristics of Agentic AI**\n\nAgentic AI systems are distinct from traditional AI mode'

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)
text_chunks=text_splitter.split_documents(documents)
text_chunks

[Document(metadata={'source': '../data/agenticAI.txt'}, page_content='**Understanding Agentic AI**'),
 Document(metadata={'source': '../data/agenticAI.txt'}, page_content='Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving'),
 Document(metadata={'source': '../data/agenticAI.txt'}, page_content='towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.'),
 Document(metadata={'source': '../data/agenticAI.txt'}, page_content='**Key Characteristics of Agentic AI**\n\nAgentic AI systems are distinct from traditional AI models due to several core characteristics:'),
 Document(metadata={'source': '../data/agenticAI.txt'}, page_content='* **Goal-Oriented:** They possess a clear objective or goal that they strive to 

In [8]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

embeddings=OpenAIEmbeddings()
vectorstore=FAISS.from_documents(text_chunks, embeddings)
vectorstore

C:\Users\meriem11\AppData\Local\Temp\ipykernel_17616\2085854719.py:4: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings=OpenAIEmbeddings()


In [9]:
retriever=vectorstore.as_retriever()

In [10]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
**Key Characteristics of Agentic AI**

Agentic AI systems are distinct from traditional AI models due to several core characteristics:
--------------------------------------------------
Document 2:
**Understanding Agentic AI**
--------------------------------------------------
Document 3:
**The Future of Agentic AI**
--------------------------------------------------
Document 4:
such as ensuring safety, interpretability, and ethical decision-making remain critical areas of research and development for the widespread adoption of agentic AI.
--------------------------------------------------


In [11]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [12]:
prompt=ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [14]:
from langchain.schema.output_parser import StrOutputParser
from langchain.chat_models import ChatOpenAI

output_parser=StrOutputParser()
llm_model=ChatOpenAI(model_name="gpt-4o-mini")

C:\Users\meriem11\AppData\Local\Temp\ipykernel_17616\4118618178.py:5: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm_model=ChatOpenAI(model_name="gpt-4o-mini")


In [15]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI represents a new paradigm in artificial intelligence, focusing on systems that operate autonomously rather than merely responding to queries or executing specific tasks. This type of AI is designed to pursue goals independently, showcasing a greater level of agency. Key characteristics of Agentic AI include enhanced autonomy, decision-making capabilities, and the ability to adapt to dynamic environments. Unlike traditional AI models, which often rely on predefined instructions, Agentic AI can autonomously navigate challenges and make judgments to achieve desired outcomes. This evolution in AI technology suggests a significant shift towards systems that can act with intention and purpose. The future of Agentic AI holds potential for more sophisticated applications across various fields, including automation, robotics, and decision support systems. Overall, Agentic AI aims to enhance the interaction between humans and machines by fostering systems that can understand and act 